# TRICARE PDF to JSON → Embedding (BAAI/bge-m3)

**한 줄 요약**
**현재 파이프라인 요약**
1. PDF → JSON 구조화
2. JSON에서 본문 추출
3. 검색 품질을 높이는 방식으로 chunk 생성
4. 다국어 임베딩
5. PDF 전용 Chroma DB 저장
6. CSV 전용 Chroma DB 저장
- 문서 전처리 → 청킹 → 임베딩 → Vector DB 저장
- tricare_llamaparse_preprocessing.ipynb 파일에서 JSON 생성
- outputs\json_docs\per_file -> 각각 PDF 파일의 JSON 파일 저장되어 있음


# 1. 라이브러리 import

In [4]:
#!pip install langchain-huggingface

In [5]:
import json
import re
from pathlib import Path

import pandas as pd

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 2. 프로젝트 파일 경로 및 DB 이름 설정
- PDF -> JSON 변환한 데이터를 쓴 모델과 쓰지 않은 모델을 비교하기 위해 이름을 다르게 설정

In [6]:
PROJECT_ROOT = Path(r"C:\SKN_24\3차 프로젝트\Care-Insure-Bot")

JSON_DIR = PROJECT_ROOT / "outputs" / "json_docs"
DATA_DIR = PROJECT_ROOT / "data"

# PDF(JSON 기반) DB
PDF_DB_DIR = PROJECT_ROOT / "tricare_pdf_json_db"
PDF_COLLECTION_NAME = "tricare_pdf_json_text"

# CSV DB
CSV_DB_DIR = PROJECT_ROOT / "tricare_csv_db"
CSV_COLLECTION_NAME = "tricare_csv_text"

print("JSON_DIR:", JSON_DIR)
print("DATA_DIR:", DATA_DIR)
print("PDF_DB_DIR:", PDF_DB_DIR)
print("CSV_DB_DIR:", CSV_DB_DIR)

JSON_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\outputs\json_docs
DATA_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data
PDF_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_pdf_json_db
CSV_DB_DIR: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_db


# 3. 공통 전처리 함수
- 공통 전처리 함수

In [7]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = str(text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def is_noise_text(text: str) -> bool:
    """
    검색 가치가 낮은 블록을 제거하기 위한 함수
    """
    if not text:
        return True

    t = text.strip()

    # 너무 짧은 텍스트
    if len(t) < 30:
        return True

    # URL만 있는 경우
    if t.startswith("http") and " " not in t:
        return True

    # 숫자 비율이 너무 높은 경우 (전화번호/주소성 블록)
    digit_ratio = sum(c.isdigit() for c in t) / max(len(t), 1)
    if digit_ratio > 0.35:
        return True

    return False

# 4. 제목 여부를 간단히 판별하는 함수
- 제목 여부를 간단히 판별하는 함수

In [8]:
def is_heading(text: str) -> bool:
    """
    간단한 제목 판별 규칙
    """
    if not text:
        return False

    t = text.strip()

    # 너무 길면 제목이 아닐 가능성 높음
    if len(t) > 120:
        return False

    # markdown heading
    if t.startswith("#"):
        return True

    # 대문자 위주 제목
    if t.isupper() and len(t) >= 5:
        return True

    return False

# 5. PDF JSON에서 block 추출
- JSON 구조에서 페이지별 단락을 가져오는 함수

In [9]:
def extract_blocks_from_json(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    source_file = data.get("file", json_path.stem)

    blocks = []

    for page_info in data.get("pages", []):
        page_num = page_info.get("page")

        for para_idx, para in enumerate(page_info.get("paragraphs", []), start=1):
            text = clean_text(para)

            if not text:
                continue

            blocks.append({
                "source_file": source_file,
                "page": page_num,
                "block_id": f"{page_num}_{para_idx}",
                "raw_text": text
            })

    return blocks

# 6. 검색 품질을 높이는 chunk 생성 함수
- 핵심 chunk 구현하는 코드

**적용**
1. 제목이면 current_heading으로 저장
2. 다음 본문 단락과 결합
3. 너무 긴 텍스트는 문장 기준으로 재분할
4. metadata 유지

In [10]:
def split_long_text(text: str, max_chars: int = 900):
    """
    긴 블록을 문장 기준으로 재분할
    """
    if len(text) <= max_chars:
        return [text]

    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)
            current = sent.strip()

    if current:
        chunks.append(current)

    return chunks


def build_pdf_documents_from_json(json_path: Path) -> list[Document]:
    blocks = extract_blocks_from_json(json_path)

    docs = []
    current_heading = None

    for block in blocks:
        text = block["raw_text"]

        if is_heading(text):
            current_heading = text
            continue

        if is_noise_text(text):
            continue

        # 제목 + 본문 결합
        combined_text = text
        if current_heading:
            combined_text = f"{current_heading}\n\n{text}"

        # 표 후보 여부
        block_type = "table" if "|" in text and "---" in text else "text"

        # 긴 블록 재분할
        sub_chunks = split_long_text(combined_text, max_chars=900)

        for sub_idx, chunk in enumerate(sub_chunks, start=1):
            if is_noise_text(chunk):
                continue

            docs.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "source_file": block["source_file"],
                        "page": block["page"],
                        "block_id": block["block_id"],
                        "sub_chunk_id": sub_idx,
                        "section_title": current_heading,
                        "block_type": block_type,
                        "data_type": "pdf_json"
                    }
                )
            )

    return docs

# 7. 전체 PDF JSON 파일을 Document로 변환
- 모든 JSON 파일을 읽어서 PDF용 Document 리스트를 만듦
- pages 수 == 0인 실패 파일은 자동 제외

In [11]:
json_files = sorted(JSON_DIR.glob("*.json"))

pdf_docs = []
skipped_json_files = []

print("JSON 파일 개수:", len(json_files))

for jf in json_files:
    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    if len(data.get("pages", [])) == 0:
        skipped_json_files.append(jf.name)
        print(f"제외됨 (pages=0): {jf.name}")
        continue

    docs = build_pdf_documents_from_json(jf)
    pdf_docs.extend(docs)
    print(f"{jf.name} → {len(docs)}개 Document")

print("\n전체 PDF Document 수:", len(pdf_docs))
print("제외된 JSON 파일 수:", len(skipped_json_files))

JSON 파일 개수: 11
ADDP_Brochure_FINAL_122624_508c.json → 38개 Document
Costs_Fees.json → 44개 Document
Maternity_Br (1).json → 37개 Document
NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).json → 110개 Document
Overseas_HB(해외 프로그램 안내서).json → 110개 Document
Pharmacy_HB(tricare 약국 프로그램 안내서).json → 227개 Document
QLEs_FS (2).json → 39개 Document
Retiring_NGR_Br.json → 40개 Document
TFL_HB(평생 트라이케어).json → 285개 Document
TOP_Handbook_AUG_2023_FINAL_092223_508 (1).json → 108개 Document
TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).json → 181개 Document

전체 PDF Document 수: 1219
제외된 JSON 파일 수: 0


# 8. PDF Document 샘플 확인

In [12]:
if pdf_docs:
    sample = pdf_docs[0]

    print("[PDF 샘플 metadata]")
    print(sample.metadata)

    print("\n[PDF 샘플 text]")
    print(sample.page_content[:700])

[PDF 샘플 metadata]
{'source_file': 'ADDP_Brochure_FINAL_122624_508c.pdf', 'page': 1, 'block_id': '1_3', 'sub_chunk_id': 1, 'section_title': '# The care you need for dental health and readiness', 'block_type': 'text', 'data_type': 'pdf_json'}

[PDF 샘플 text]
# The care you need for dental health and readiness

The Active Duty Dental Program provides civilian dental care for active duty service members. The ADDP helps ensure your dental health and deployment readiness. United Concordia Companies, Inc. administers the ADDP benefit. Visit www.addp-ucci.com for more information.


# 9. CSV 한 행을 자연어 텍스트로 변환

In [13]:
def row_to_text(row: pd.Series) -> str:
    parts = []

    for col, val in row.items():
        if pd.isna(val):
            continue

        val_str = str(val).strip()
        if not val_str:
            continue

        parts.append(f"{col}: {val_str}")

    return "\n".join(parts)

# 10. CSV를 Document로 변환
- CSV 파일 1개를 row 단위 Document 리스트

In [14]:
def csv_file_to_documents(csv_path: Path) -> list[Document]:
    df = pd.read_csv(csv_path)

    docs = []

    for idx, row in df.iterrows():
        text = clean_text(row_to_text(row))

        if is_noise_text(text):
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source_file": csv_path.name,
                    "row_index": int(idx),
                    "data_type": "csv",
                    "block_type": "row"
                }
            )
        )

    return docs

# 11. 사용할 CSV 목록 지정
- 프로젝트에서 사용할 CSV 경로를 지정

In [15]:
csv_paths = [
    DATA_DIR / "mental_health_services.csv",
    DATA_DIR / "TricarePlans.csv",
    DATA_DIR / "tricare_exclusions.csv",
    DATA_DIR / "Health_Plan_Costs.csv"
]

for p in csv_paths:
    print(p.name, "존재:", p.exists())

mental_health_services.csv 존재: True
TricarePlans.csv 존재: True
tricare_exclusions.csv 존재: True
Health_Plan_Costs.csv 존재: True


# 12. 전체 CSV를 Document로 변환
- CSV 파일들을 모두 읽어서 CSV용 Document 리스트를 생성

In [16]:
csv_docs = []

for csv_path in csv_paths:
    if not csv_path.exists():
        print("없음:", csv_path.name)
        continue

    docs = csv_file_to_documents(csv_path)
    csv_docs.extend(docs)
    print(f"{csv_path.name} → {len(docs)}개 Document")

print("\n전체 CSV Document 수:", len(csv_docs))

mental_health_services.csv → 8개 Document
TricarePlans.csv → 10개 Document
tricare_exclusions.csv → 55개 Document
Health_Plan_Costs.csv → 234개 Document

전체 CSV Document 수: 307


# 13. CSV Document 샘플 확인
- CSV 변환 결과를 점검하는 코드

In [17]:
if csv_docs:
    sample = csv_docs[0]

    print("[CSV 샘플 metadata]")
    print(sample.metadata)

    print("\n[CSV 샘플 text]")
    print(sample.page_content[:500])

[CSV 샘플 metadata]
{'source_file': 'mental_health_services.csv', 'row_index': 0, 'data_type': 'csv', 'block_type': 'row'}

[CSV 샘플 text]
서비스: Psychotherapy
내용: Psychotherapy is discussion-based mental health therapy. It’s also known as “therapy” and “counseling.” TRICARE covers office-based outpatient psychotherapy when medically or psychologically necessary to treat a mental health disorder. TRICARE covers inpatient and outpatient services, including: Individual sessions: Individual sessions are covered up to 60 minutes. Individual crisis sessions may extend up to 120 minutes. Family or conjoint sessions: Therapy designed to tre


# 14. 다국어 임베딩 모델 로드
- PDF/CSV 모두에 사용할 임베딩 모델을 로드

In [18]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("임베딩 모델 로드 완료")


임베딩 모델 로드 완료


# 15. PDF 전용 Chroma DB 생성

In [22]:
vectorstore_pdf = Chroma.from_documents(
    documents=pdf_docs,
    embedding=embedding_model,
    persist_directory=str(PDF_DB_DIR),
    collection_name=PDF_COLLECTION_NAME
)

print("PDF 전용 Chroma DB 생성 완료")
print("저장 경로:", PDF_DB_DIR)
print("컬렉션 이름:", PDF_COLLECTION_NAME)

PDF 전용 Chroma DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_pdf_json_db
컬렉션 이름: tricare_pdf_json_text


# 16. CSV 전용 Chroma DB 생성
- CSV 기반 Document만 저장
- DB 경로: tricare_csv_db
- 컬렉션 이름: tricare_csv_text

In [19]:
vectorstore_csv = Chroma.from_documents(
    documents=csv_docs,
    embedding=embedding_model,
    persist_directory=str(CSV_DB_DIR),
    collection_name=CSV_COLLECTION_NAME
)

print("CSV 전용 Chroma DB 생성 완료")
print("저장 경로:", CSV_DB_DIR)
print("컬렉션 이름:", CSV_COLLECTION_NAME)

CSV 전용 Chroma DB 생성 완료
저장 경로: C:\SKN_24\3차 프로젝트\Care-Insure-Bot\tricare_csv_db
컬렉션 이름: tricare_csv_text


# 17. PDF DB 검색 테스트 (품질 확인)
- PDF DB 검색 품질을 확인하는 코드

In [23]:
query = "Who is eligible for the Active Duty Dental Program?"

results = vectorstore_pdf.similarity_search(query, k=3)

for i, r in enumerate(results, start=1):
    print(f"\n=== PDF 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])


=== PDF 검색 결과 1 ===
metadata: {'section_title': '# CHANGE IN FEDERAL EMPLOYEES HEALTH BENEFITS PROGRAM ELIGIBILITY OR ENROLLMENT', 'page': 14, 'block_id': '14_2', 'block_type': 'table', 'source_file': 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'data_type': 'pdf_json', 'sub_chunk_id': 1}
text: # CHANGE IN FEDERAL EMPLOYEES HEALTH BENEFITS PROGRAM ELIGIBILITY OR ENROLLMENT

| PLAN | WHO’S ELIGIBLE | DETAILS |
| ----------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------

=== PDF 검색 결과 2 ===
metadata: {'block_type': 'text', 'page': 3, 'section_title': '# The Care You Need for Dental Readiness', 'block_id': '3_3', 'source_file': 'TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서)

# 18. CSV DB 검색 테스트 (품질 확인)
- CSV DB 검색 품질을 확인

In [24]:
query = "mental health counseling coverage"

results = vectorstore_csv.similarity_search(query, k=3)

for i, r in enumerate(results, start=1):
    print(f"\n=== CSV 검색 결과 {i} ===")
    print("metadata:", r.metadata)
    print("text:", r.page_content[:500])


=== CSV 검색 결과 1 ===
metadata: {'block_type': 'row', 'data_type': 'csv', 'row_index': 15, 'source_file': 'tricare_exclusions.csv'}
text: name: Elective Psychotherapy and Mind Expansion Psychotherapy
title: Elective Psychotherapy and Mind Expansion Psychotherapy
content: TRICARE doesn’t cover: Last Updated 3/20/2022
url: https://www.tricare.mil/CoveredServices/IsItCovered/ElectivePsychotherapy

=== CSV 검색 결과 2 ===
metadata: {'source_file': 'tricare_exclusions.csv', 'data_type': 'csv', 'block_type': 'row', 'row_index': 30}
text: name: Mental Health Exclusions
title: Mental Health Exclusions
content: TRICARE doesn’t cover the following services: There may be other exclusions that aren’t on this list. To learn more about exclusions,call your regional contractor. Last Updated 1/30/2026
url: https://www.tricare.mil/CoveredServices/IsItCovered/MentalHealthExclusions

=== CSV 검색 결과 3 ===
metadata: {'block_type': 'row', 'data_type': 'csv', 'row_index': 0, 'source_file': 'mental_health_services.